In [ ]:
# %pip install ipympl

In [ ]:
# %matplotlib widget
from pathlib import Path
from datetime import datetime
import sys
import time
import base64
import html

import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

# Import z pliku pomocniczego
from derma_helpers import (
    image_info, card, filter_detections, bbox_iou, 
    find_common_detections, draw_detections, build_legend,
    DISPLAY_CLASSES, V4_CLASS_NAMES, V4_CLASS_MAP
)


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predict import run_inference, load_model, load_sahi_model
from config import MODEL_v3, MODEL_v4, RUNS_DIR, CONF_THRESHOLD

# IMAGES_DIR = PROJECT_ROOT / "images" / "zuza" / "dated_fotoes"
IMAGES_DIR = PROJECT_ROOT / "images" / "my" / "dermaai" / "dated_photos"
# Models\yolo\images\my\dermaai\dated_photos

if not IMAGES_DIR.exists():
    raise FileNotFoundError(f"Image directory not found: {IMAGES_DIR}")

images = sorted(
    p for p in IMAGES_DIR.rglob("*")
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}
)

if not images:
    raise FileNotFoundError(f"No images found in {IMAGES_DIR}")

MODEL_v3 = Path(MODEL_v3)
MODEL_v4 = Path(MODEL_v4)


BG = "#0B1220"
PANEL = "#111827"
CARD = "#1E293B"
BORDER = "#334155"
TEXT = "#F8FAFC"
MUTED = "#94A3B8"

AVAILABLE_MODELS = {
    "v3.2 — Acne (class 0)": str(MODEL_v3),
    "v4 — Acne classes only": str(MODEL_v4),
    "Both models — comparison": "both",
    "Common detections — v3.2 ∩ v4": "intersection",
}


model_select = widgets.Dropdown(options=list(AVAILABLE_MODELS.items()), value="both", description="Model", layout=widgets.Layout(width="100%"), style={"description_width": "100px"})
image_select = widgets.Dropdown(options=[(f"{image_info(p, IMAGES_DIR)[0]} | {image_info(p, IMAGES_DIR)[1]} | {p.name}", p) for p in images], description="Image", layout=widgets.Layout(width="100%"), style={"description_width": "100px"})
conf_slider = widgets.FloatSlider(value=float(CONF_THRESHOLD), min=0.05, max=0.95, step=0.05, description="Confidence", continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="100%"), style={"description_width": "100px"})
match_iou_slider = widgets.FloatSlider(value=0.30, min=0.05, max=0.95, step=0.05, description="Match IoU", continuous_update=False, readout_format=".2f", layout=widgets.Layout(width="100%"), style={"description_width": "100px"})
mode_select = widgets.Dropdown(options=[("Standard", "standard"), ("TTA", "tta"), ("SAHI", "sahi"), ("SAHI + TTA", "sahi_tta")], value="standard", description="Inference", layout=widgets.Layout(width="100%"), style={"description_width": "100px"})
preprocess_check = widgets.Checkbox(value=True, description="Detail enhancement / preprocessing", indent=False, layout=widgets.Layout(width="100%"))
run_button = widgets.Button(description="RUN PREDICTION", icon="play", layout=widgets.Layout(width="100%", height="42px"))
output = widgets.Output()

title = widgets.HTML(f"""<div style="background:{BG};border:1px solid {BORDER};border-radius:12px;padding:18px 22px;margin-bottom:12px;"><div style="color:{TEXT};font-size:22px;font-weight:700;">ACNE DETECTION</div><div style="color:{MUTED};font-size:12px;margin-top:4px;">YOLO inference control panel</div></div>""")

controls = widgets.VBox([
    widgets.HTML(f"""<div style="color:{MUTED};font-size:10px;font-weight:700;letter-spacing:1px;margin-bottom:8px;">INFERENCE SETTINGS</div>"""),
    model_select, image_select, conf_slider, match_iou_slider, mode_select,
    widgets.HTML("<div style='height:6px'></div>"), preprocess_check,
    widgets.HTML("<div style='height:8px'></div>"), run_button
], layout=widgets.Layout(width="460px", padding="18px", border=f"1px solid {BORDER}", border_radius="12px", background_color=PANEL))


def run_prediction(_):
    run_button.disabled = True
    run_button.description = "PROCESSING..."

    with output:
        clear_output(wait=True)
        try:
            image_path = Path(image_select.value)
            selection = model_select.value
            confidence = float(conf_slider.value)
            match_iou = float(match_iou_slider.value)
            mode = mode_select.value
            preprocess = preprocess_check.value

            image_date, image_side = image_info(image_path, IMAGES_DIR)
            tta = mode in ("tta", "sahi_tta")
            sahi = mode in ("sahi", "sahi_tta")

            if selection in ("both", "intersection"):
                jobs = [("v3.2", MODEL_v3), ("v4", MODEL_v4)]
            elif Path(selection) == MODEL_v3:
                jobs = [("v3.2", MODEL_v3)]
            else:
                jobs = [("v4", MODEL_v4)]

            for version, model_path in jobs:
                if not model_path.exists():
                    raise FileNotFoundError(f"Model {version} not found: {model_path}")

            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            import predict as predict_module
            old_conf = predict_module.CONF_THRESHOLD

            model_results = {}
            session_dirs = []
            detections_by_model = {"v3.2": [], "v4": []}

            try:
                predict_module.CONF_THRESHOLD = confidence
                for version, model_path in jobs:
                    session_dir = RUNS_DIR / "notebook" / f"predict_{model_path.stem}_{mode}_{timestamp}"
                    session_dir.mkdir(parents=True, exist_ok=True)
                    session_dirs.append(session_dir)

                    model = load_sahi_model(model_path=model_path, confidence_threshold=confidence) if sahi else load_model(model_path=model_path)
                    start_time = time.time()
                    
                    result = run_inference(image_path=image_path, model=model, session_dir=session_dir, preprocess=preprocess, tta=tta, sahi=sahi)
                    elapsed = time.time() - start_time

                    if result is None:
                        raise RuntimeError(f"Inference returned no result for {model_path.name}")

                    model_results[version] = {**result, "elapsed": elapsed}
                    detections_by_model[version] = filter_detections(result["detections"], version, V4_CLASS_NAMES, V4_CLASS_MAP, DISPLAY_CLASSES)
            finally:
                predict_module.CONF_THRESHOLD = old_conf

            if selection == "intersection":
                all_detections = find_common_detections(detections_by_model["v3.2"], detections_by_model["v4"], match_iou)
                view_title = f"COMMON DETECTIONS | Match IoU ≥ {match_iou:.2f}"
            elif selection == "both":
                all_detections = detections_by_model["v3.2"] + detections_by_model["v4"]
                view_title = "BOTH MODELS — ALL ACNE DETECTIONS"
            elif Path(selection) == MODEL_v3:
                all_detections = detections_by_model["v3.2"]
                view_title = "MODEL v3 — ACNE"
            else:
                all_detections = detections_by_model["v4"]
                view_title = "MODEL v4 — ACNE CLASSES"

            original_bgr = cv2.imread(str(image_path))
            if original_bgr is None:
                raise ValueError(f"Cannot read image: {image_path}")
            original_rgb = cv2.cvtColor(original_bgr, cv2.COLOR_BGR2RGB)

            # annotated = draw_detections(original_rgb, all_detections, DISPLAY_CLASSES)
            annotated = draw_detections(
                original_rgb, 
                all_detections, 
                display_classes=DISPLAY_CLASSES, 
                dynamic_scale=True, 
                show_class_name=True, 
                show_confidence=True,
                image_is_rgb=True
            )
            detection_count = len(all_detections)
            avg_conf = sum(float(d["confidence"]) for d in all_detections) / detection_count if detection_count else 0.0

            if len(model_results) == 1:
                version, result = next(iter(model_results.items()))
                score_text = f"{version}: {result['score']:.3f}"
            else:
                score_text = f"v3.2: {model_results['v3.2']['score']:.3f} | v4: {model_results['v4']['score']:.3f}"

            total_time = sum(result["elapsed"] for result in model_results.values())

            fig, (ax_img, ax_legend) = plt.subplots(1, 2, figsize=(16, 10), gridspec_kw={"width_ratios": [3.4, 1.5]})
            fig.patch.set_facecolor(BG)
            ax_img.set_facecolor(BG)
            ax_legend.set_facecolor(PANEL)

            ax_img.imshow(annotated)
            ax_img.axis("off")
            ax_img.set_title(f"{view_title}\n{image_date} | {image_side.upper()} | {image_path.name}", color=TEXT, fontsize=11, fontweight="bold", pad=12)

            ax_legend.axis("off")
            ax_legend.set_title("LEGENDA KLAS", color=TEXT, fontsize=12, fontweight="bold", loc="left", pad=12)

            legend_handles = build_legend(all_detections, DISPLAY_CLASSES)
            if legend_handles:
                ax_legend.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(0.0, 0.96), frameon=True, facecolor=PANEL, edgecolor=BORDER, labelcolor=TEXT, fontsize=9, handlelength=1.5, handleheight=1.5, borderpad=1.2, labelspacing=1.5)
            else:
                ax_legend.text(0.02, 0.90, "Brak wspólnych detekcji." if selection == "intersection" else "Nie wykryto zmian trądzikowych.", transform=ax_legend.transAxes, color=MUTED, fontsize=10, va="top")

            plt.tight_layout()

            selection_label = {str(MODEL_v3): "v3.2", str(MODEL_v4): "v4", "both": "both", "intersection": "intersection"}.get(selection, "model")
            visualization_dir = RUNS_DIR / "notebook" / f"visualization_{selection_label}_{mode}_{timestamp}"
            visualization_dir.mkdir(parents=True, exist_ok=True)

            combined_path = visualization_dir / "annotated_with_legend.png"
            image_only_path = visualization_dir / "annotated_image.jpg"

            fig.savefig(combined_path, dpi=180, bbox_inches="tight", facecolor=fig.get_facecolor())
            cv2.imwrite(str(image_only_path), cv2.cvtColor(annotated, cv2.COLOR_RGB2BGR))
            plt.close(fig)

            with open(combined_path, "rb") as img_file:
                img_base64 = base64.b64encode(img_file.read()).decode("utf-8")

            model_label = {"both": "Both models", "intersection": "Common detections"}.get(selection, Path(selection).stem)
            class_counts = {class_id: sum(det["display_id"] == class_id for det in all_detections) for class_id in DISPLAY_CLASSES}
            
            class_cards = "".join(card(f"ID {class_id}: {DISPLAY_CLASSES[class_id]['name']}", class_counts[class_id], CARD, BORDER, TEXT, MUTED) for class_id in DISPLAY_CLASSES if class_counts[class_id] > 0)
            output_dirs_html = "".join(f'<div style="font-size:9px;word-break:break-all;margin-bottom:6px;">{html.escape(str(path))}</div>' for path in (session_dirs + [visualization_dir]))
            match_info = card("MATCH IoU", f"{match_iou:.2f}", CARD, BORDER, TEXT, MUTED) if selection == "intersection" else ""

            summary_html = f"""
            <div style="background:{PANEL};border:1px solid {BORDER};border-radius:12px;padding:18px;width:100%;max-width:950px;box-sizing:border-box;font-family:Arial,sans-serif;color:{TEXT};">
                <div style="margin-bottom: 20px; text-align: center;"><img src="data:image/png;base64,{img_base64}" style="width: 100%; border-radius: 8px; border: 1px solid {BORDER};" /></div>
                <div style="font-size:17px;font-weight:700;">ANALYSIS</div>
                <div style="color:{MUTED};font-size:10px;margin:5px 0 15px;">Displayed acne detections</div>
                <div style="display:grid;grid-template-columns:repeat(4,minmax(0,1fr));gap:8px;">
                    {card("MODEL / VIEW", model_label, CARD, BORDER, TEXT, MUTED)}
                    {card("DETECTIONS", detection_count, CARD, BORDER, TEXT, MUTED)}
                    {card("AVG CONF.", f"{avg_conf:.3f}", CARD, BORDER, TEXT, MUTED)}
                    {card("INFERENCE TIME", f"{total_time:.2f}s", CARD, BORDER, TEXT, MUTED)}
                    {card("PIPELINE SCORE", score_text, CARD, BORDER, TEXT, MUTED)}
                    {card("CONFIDENCE", f"{confidence:.2f}", CARD, BORDER, TEXT, MUTED)}
                    {card("INFERENCE", mode.upper(), CARD, BORDER, TEXT, MUTED)}
                    {match_info}
                </div>

                <div style="border-top:1px solid {BORDER};margin-top:16px;padding-top:14px;">
                    <div style="color:{MUTED};font-size:9px;font-weight:700;letter-spacing:1px;margin-bottom:9px;">COUNTS BY CLASS</div>
                    <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:8px;">
                        {class_cards or card("DETECTIONS", "None", CARD, BORDER, TEXT, MUTED)}
                    </div>
                </div>

                <div style="border-top:1px solid {BORDER};margin-top:16px;padding-top:14px;">
                    <div style="color:{MUTED};font-size:9px;font-weight:700;letter-spacing:1px;margin-bottom:9px;">PIPELINE SETTINGS</div>
                    <div style="display:grid;grid-template-columns:repeat(3,minmax(0,1fr));gap:8px;">
                        {card("PREPROCESS", "ON" if preprocess else "OFF", CARD, BORDER, TEXT, MUTED)}
                        {card("TTA", "ON" if tta else "OFF", CARD, BORDER, TEXT, MUTED)}
                        {card("SAHI", "ON" if sahi else "OFF", CARD, BORDER, TEXT, MUTED)}
                    </div>
                </div>

                <div style="border-top:1px solid {BORDER};margin-top:16px;padding-top:12px;">
                    <div style="color:{MUTED};font-size:9px;font-weight:700;letter-spacing:1px;margin-bottom:8px;">SAVED OUTPUTS</div>
                    <div style="font-size:10px;word-break:break-all;margin-bottom:8px;">Combined image + legend: {html.escape(str(combined_path))}</div>
                    <div style="font-size:10px;word-break:break-all;margin-bottom:8px;">Annotated image only: {html.escape(str(image_only_path))}</div>
                    {output_dirs_html}
                </div>
            </div>
            """
            display(widgets.HTML(value=summary_html))

        except Exception as exc:
            print(f"Prediction failed: {type(exc).__name__}: {exc}")
            raise
        finally:
            run_button.disabled = False
            run_button.description = "RUN PREDICTION"

run_button._click_handlers.callbacks.clear()
run_button.on_click(run_prediction)

ui = widgets.VBox([title, controls, output])
display(ui);

In [ ]:
from pathlib import Path
import sys
from datetime import datetime
import cv2
import pandas as pd
import json

from derma_helpers import (image_info, draw_detections, filter_detections, create_skin_mask, 
                           DISPLAY_CLASSES, V4_CLASS_NAMES, V4_CLASS_MAP)


# PATHS AND IMPORTS

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predict import run_inference, load_model, load_sahi_model

from config import MODEL_v3, MODEL_v4, BEST_MODEL, RUNS_DIR, CONF_THRESHOLD

MODEL_CHOICE = "v4"

if MODEL_CHOICE == "v3.2":
    SELECTED_MODEL = MODEL_v3
elif MODEL_CHOICE == "v4":
    SELECTED_MODEL = MODEL_v4
else:
    SELECTED_MODEL = BEST_MODEL

SELECTED_MODEL = Path(SELECTED_MODEL)

IMAGES_DIR = PROJECT_ROOT / "images" / "zuza" / "dated_fotoes"


CONFIDENCE = 0.20
NMS_IOU = 0.15
MODE = "tta"  # "standard", "tta", "sahi", "sahi_tta"
PREPROCESS = True

SAVE_RESULTS_JSON = True
SAVE_RESULTS_CSV = True
SAVE_PREDICTED_IMAGES = True
PREDICTED_IMAGE_FORMAT = "jpg"
PREDICTED_BOX_THICKNESS = 4
SHOW_CONFIDENCE = True
SHOW_CLASS_NAME = True
PREDICTED_BOX_COLOR = (0, 0, 255)
PREDICTED_FONT_SCALE = 0.75
PREDICTED_TEXT_THICKNESS = 1

OUTPUT_DIR = RUNS_DIR / "batch"


VALID_SIDES = {"front", "right", "left"}
VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

images = []
for p in IMAGES_DIR.rglob("*"):
    if p.suffix.lower() not in VALID_EXTENSIONS:
        continue
    image_date, image_side = image_info(p, IMAGES_DIR)
    if image_side not in VALID_SIDES:
        continue
    images.append(p)

images = sorted(images)
if not images:
    raise FileNotFoundError(f"No front/right/left images found in {IMAGES_DIR}")

print(f"Found {len(images)} images")


tta = MODE in ["tta", "sahi_tta"]
sahi = MODE in ["sahi", "sahi_tta"]

model = load_sahi_model() if sahi else load_model()

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
session_dir = OUTPUT_DIR / f"predict_{SELECTED_MODEL.stem}_{MODE}_{timestamp}"
session_dir.mkdir(parents=True, exist_ok=True)

predicted_dir = session_dir / "predicted"
if SAVE_PREDICTED_IMAGES:
    predicted_dir.mkdir(parents=True, exist_ok=True)

print(f"Mode: {MODE} | Confidence: {CONFIDENCE} | Output: {session_dir}\n")

import predict as predict_module
old_conf = predict_module.CONF_THRESHOLD
old_iou = predict_module.NMS_IOU_THRESHOLD

predict_module.CONF_THRESHOLD = CONFIDENCE
predict_module.NMS_IOU_THRESHOLD = NMS_IOU

results = []

try:
    for i, image_path in enumerate(images, start=1):
        image_date, image_side = image_info(image_path, IMAGES_DIR)
        print(f"[{i}/{len(images)}] {image_date} | {image_side} | {image_path.name}")

        try:
            result = run_inference(
                image_path=image_path,
                model=model,
                session_dir=session_dir,
                preprocess=PREPROCESS,
                tta=tta,
                sahi=sahi
            )

            if result is None:
                print("  -> ERROR: no result")
                continue

            detections = result["detections"]
            acne_count = len(detections)
            avg_conf = sum(d["confidence"] for d in detections) / acne_count if acne_count else 0.0

            predicted_path = None
            if SAVE_PREDICTED_IMAGES:
                image = cv2.imread(str(image_path))
                if image is not None:

                    skin_mask = create_skin_mask(image)
                        
                    raw_filtered = filter_detections(
                        detections, 
                        MODEL_CHOICE, 
                        V4_CLASS_NAMES, 
                        V4_CLASS_MAP, 
                        DISPLAY_CLASSES
                    )

                    filtered_detections = []
                    for det in raw_filtered:
                        bbox = det.get("bbox")
                        if bbox and len(bbox) == 4:
                            cx, cy, _, _ = map(float, bbox)
                            ix, iy = int(round(cx)), int(round(cy))
                            
                            h, w = skin_mask.shape[:2]
                            if 0 <= ix < w and 0 <= iy < h:
                                if skin_mask[iy, ix] > 0:
                                    filtered_detections.append(det)
                            else:
                                pass
                        else:
                            filtered_detections.append(det)

                    predicted_image = draw_detections(
                        image, 
                        filtered_detections, 
                        display_classes=DISPLAY_CLASSES,
                        default_color=PREDICTED_BOX_COLOR, 
                        thickness=PREDICTED_BOX_THICKNESS,
                        font_scale=PREDICTED_FONT_SCALE, 
                        text_thickness=PREDICTED_TEXT_THICKNESS,
                        show_class_name=SHOW_CLASS_NAME, 
                        show_confidence=SHOW_CONFIDENCE,
                        image_is_rgb=False
                    )
                    
                    predicted_path = predicted_dir / image_date / image_side / image_path.name
                    predicted_path.parent.mkdir(parents=True, exist_ok=True)

                    if PREDICTED_IMAGE_FORMAT.lower() != image_path.suffix.lower().replace(".", ""):
                        predicted_path = predicted_path.with_suffix("." + PREDICTED_IMAGE_FORMAT)

                    cv2.imwrite(str(predicted_path), predicted_image)
                    print(f"  -> saved: {predicted_path}")

            results.append({
                "date": image_date,
                "side": image_side,
                "filename": image_path.name,
                "image_path": str(image_path),
                "predicted_image_path": str(predicted_path) if predicted_path else None,
                "score": result["score"],
                "acne_count": acne_count,
                "avg_conf": avg_conf,
                "time": result["time"],
                "confidence_threshold": CONFIDENCE,
                "nms_iou": NMS_IOU,
                "mode": MODE,
                "preprocess": PREPROCESS,
                "tta": tta,
                "sahi": sahi,
                "model": str(SELECTED_MODEL),
            })
            print(f"  -> score={result['score']:.4f}, count={acne_count}, avg_conf={avg_conf:.4f}")

        except Exception as e:
            print(f"  -> ERROR: {e}")
            results.append({
                "date": image_date, "side": image_side, "filename": image_path.name,
                "image_path": str(image_path), "predicted_image_path": None,
                "score": None, "acne_count": None, "avg_conf": None, "time": None,
                "confidence_threshold": CONFIDENCE, "nms_iou": NMS_IOU,
                "mode": MODE, "preprocess": PREPROCESS, "tta": tta, "sahi": sahi,
                "model": str(SELECTED_MODEL), "error": str(e),
            })
finally:
    predict_module.CONF_THRESHOLD = old_conf
    predict_module.NMS_IOU_THRESHOLD = old_iou


df = pd.DataFrame(results)
df = df.sort_values(["date", "side", "filename"]).reset_index(drop=True)

if SAVE_RESULTS_JSON:
    json_path = session_dir / "results.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"JSON: {json_path}")

if SAVE_RESULTS_CSV:
    csv_path = session_dir / "results.csv"
    df.to_csv(csv_path, index=False)
    print(f"CSV:  {csv_path}")

print("\n" + "="*60 + "\nBATCH INFERENCE COMPLETE\n" + "="*60)
display(df.head())

## ANLYSIS & VISUALIZATIONS 

In [ ]:
from pathlib import Path
import sys
from datetime import datetime
import json
import cv2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#bdc3c7'
plt.rcParams['axes.linewidth'] = 0.8


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predict import run_inference, load_model, load_sahi_model
from config import BEST_MODEL, RUNS_DIR, CONF_THRESHOLD

IMAGES_DIR = PROJECT_ROOT / "images" / "my" / "dermaai" / "dated_photos"
OUTPUT_DIR = RUNS_DIR / "batch"


df = None
session_dir = None

existing_csvs = list(OUTPUT_DIR.glob("*/results.csv")) if OUTPUT_DIR.exists() else []

print(f"[LOG] Checking for existing results in directory: {OUTPUT_DIR.resolve()}")

if OUTPUT_DIR.exists():
    print(f"[LOG] Directory exists: {OUTPUT_DIR}")
else:
    print(f"[LOG] Directory does not exist (will be created): {OUTPUT_DIR}")

In [ ]:
if existing_csvs:
    latest_csv = max(existing_csvs, key=lambda p: p.stat().st_mtime)
    session_dir = latest_csv.parent
    print(f"[LOG] Found existing session directory: {session_dir.resolve()}")
    print(f"[LOG] Loading ready results from CSV file (skipping re-inference).")
    df = pd.read_csv(latest_csv)
else:
    print(f"[LOG] No previous results found. Starting inference process from scratch...")
    
    # INFERENCE PARAMETERS
    CONFIDENCE = 0.10
    NMS_IOU = 0.15
    MODE = "tta"  # "standard", "tta", "sahi", "sahi_tta"
    PREPROCESS = True

    SAVE_RESULTS_JSON = True
    SAVE_RESULTS_CSV = True
    SAVE_PREDICTED_IMAGES = True
    PREDICTED_BOX_THICKNESS = 2
    SHOW_CONFIDENCE = True
    SHOW_CLASS_NAME = True
    PREDICTED_BOX_COLOR = (41, 128, 185) # Modern blue
    PREDICTED_FONT_SCALE = 0.55
    PREDICTED_TEXT_THICKNESS = 1

    def image_info(path):
        parts = path.relative_to(IMAGES_DIR).parts
        date = parts[0] if len(parts) > 1 else "Unknown"
        side = parts[1].lower() if len(parts) > 2 else "Unknown"
        return date, side

    VALID_SIDES = {"front", "right", "left"}
    VALID_EXTENSIONS = {".jpg", ".jpeg", ".png"}

    images = []
    for p in IMAGES_DIR.rglob("*"):
        if p.suffix.lower() not in VALID_EXTENSIONS:
            continue
        image_date, image_side = image_info(p)
        if image_side not in VALID_SIDES:
            continue
        images.append(p)

    images = sorted(images)
    if not images:
        raise FileNotFoundError(f"No front/right/left images found in {IMAGES_DIR}")

    print(f"Found images to process: {len(images)}")

    tta = MODE in ["tta", "sahi_tta"]
    sahi = MODE in ["sahi", "sahi_tta"]
    model = load_sahi_model() if sahi else load_model()

    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
    session_dir = OUTPUT_DIR / f"predict_{BEST_MODEL.stem}_{MODE}_{timestamp}"
    session_dir.mkdir(parents=True, exist_ok=True)

    predicted_dir = session_dir / "predicted"
    if SAVE_PREDICTED_IMAGES:
        predicted_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n[LOG] Main results session directory: {session_dir.resolve()}")

    import predict as predict_module
    old_conf = predict_module.CONF_THRESHOLD
    old_iou = predict_module.NMS_IOU_THRESHOLD
    predict_module.CONF_THRESHOLD = CONFIDENCE
    predict_module.NMS_IOU_THRESHOLD = NMS_IOU

    results = []
    try:
        for i, image_path in enumerate(images, start=1):
            image_date, image_side = image_info(image_path)
            print(f"\n[{i}/{len(images)}] Processing: {image_date} | {image_side} | {image_path.name}")
            try:
                result = run_inference(
                    image_path=image_path, model=model, session_dir=session_dir,
                    preprocess=PREPROCESS, tta=tta, sahi=sahi
                )
                if result is None:
                    print("  -> ERROR: No inference result.")
                    continue

                detections = result["detections"]
                acne_count = len(detections)
                avg_conf = sum(d["confidence"] for d in detections) / acne_count if acne_count else 0.0

                predicted_path = None
                if SAVE_PREDICTED_IMAGES:
                    image = cv2.imread(str(image_path))
                    if image is not None:
                        # Simple drawing function (kept from original)
                        output = image.copy()
                        for detection in detections:
                            bbox = detection.get("bbox")
                            if bbox is None or len(bbox) != 4:
                                continue
                            cx, cy, bw, bh = map(float, bbox)
                            x1, y1 = int(round(cx - bw / 2)), int(round(cy - bh / 2))
                            x2, y2 = int(round(cx + bw / 2)), int(round(cy + bh / 2))
                            h, w = output.shape[:2]
                            x1, y1 = max(0, min(x1, w - 1)), max(0, min(y1, h - 1))
                            x2, y2 = max(0, min(x2, w - 1)), max(0, min(y2, h - 1))
                            cv2.rectangle(output, (x1, y1), (x2, y2), PREDICTED_BOX_COLOR, PREDICTED_BOX_THICKNESS)
                        
                        predicted_path = predicted_dir / image_date / image_side / image_path.name
                        predicted_path.parent.mkdir(parents=True, exist_ok=True)
                        cv2.imwrite(str(predicted_path), output)

                record = {
                    "date": image_date,
                    "side": image_side,
                    "filename": image_path.name,
                    "image_path": str(image_path),
                    "predicted_image_path": str(predicted_path) if predicted_path else None,
                    "score": result["score"],
                    "acne_count": acne_count,
                    "avg_conf": avg_conf,
                    "time": result["time"],
                    "model": str(BEST_MODEL),
                }
                results.append(record)
            except Exception as e:
                print(f"  -> EXCEPTION: {e}")
    finally:
        predict_module.CONF_THRESHOLD = old_conf
        predict_module.NMS_IOU_THRESHOLD = old_iou

    df = pd.DataFrame(results)
    df = df.sort_values(["date", "side", "filename"]).reset_index(drop=True)

    if SAVE_RESULTS_JSON:
        with open(session_dir / "results.json", "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)
    if SAVE_RESULTS_CSV:
        df.to_csv(session_dir / "results.csv", index=False)

    print(f"\n[LOG] Inference parameters summary:")
    print(f"  - CONFIDENCE: {CONFIDENCE}")
    print(f"  - NMS_IOU: {NMS_IOU}")
    print(f"  - MODE: {MODE}")
    print(f"  - PREPROCESS: {PREPROCESS}")
    print(f"  - TTA: {tta}")
    print(f"  - SAHI: {sahi}")
    print(f"  - MODEL: {BEST_MODEL}")

## CZYNNIKI TRĄDZIKOWE

In [ ]:
import pandas as pd 
from pathlib import Path

BASE_DIR = Path(r"../images/zuza")
EXCEL_PATH = BASE_DIR / "tradzik_tabelka.xlsx"


try:
    dieta = pd.read_excel(EXCEL_PATH, sheet_name="dieta")
    rano = pd.read_excel(EXCEL_PATH, sheet_name="rano")
    wieczor = pd.read_excel(EXCEL_PATH, sheet_name="wieczór")
except Exception as e:
    raise RuntimeError(f"Mamy problem z wczytaniem pliku: {e}")

for df_temp in [dieta, rano, wieczor]:
    df_temp.columns = df_temp.columns.str.strip()

wieczor = wieczor.rename(columns={
    "data": "date",
    "krem nawilżający": "krem_nawilzajacy_wieczor",
    "antybiotyk": "antybiotyk",
    "kolagen": "kolagen"
})

rano = rano.rename(columns={
    "data": "date",
    "krem nawilżający": "krem_nawilzajacy_rano",
    "SPF": "spf"
})

dieta = dieta.rename(columns={
    "data": "date",
    "stress": "stres",
    "zbyt duża ilość słodyczy (cukru)": "duzo_cukru",
    "fast food/ gotowce": "fast_food",
    "alkohol": "alkohol",
    "smażone potrawy (zawierają dużo tłuszczów)": "smazone",
    "papierosy": "papierosy",
    "produkty pszenne": "pszenica",
    "mleko/ produkty mlecze (w dużej ilości)": "duzo_nabialu",
    "ostre potrawy": "ostre_jedzenie",
    "zbyt mała ilość wody": "malo_wody"
})

for df_temp in [wieczor, rano, dieta]:
    df_temp["date"] = pd.to_datetime(df_temp["date"], errors="coerce").dt.normalize()

# Zakres dat ze wszystkich źródeł
min_date = min(df_temp["date"].min() for df_temp in [wieczor, rano, dieta])
max_date = max(df_temp["date"].max() for df_temp in [wieczor, rano, dieta])

print(f"Zakres dat: {min_date:%Y-%m-%d} — {max_date:%Y-%m-%d}")


df_combined = wieczor.merge(rano, on="date", how="outer").merge(dieta, on="date", how="outer")

all_dates = pd.date_range(min_date, max_date, freq="D")
df_combined = df_combined.set_index("date").reindex(all_dates)
df_combined.index.name = "date"
df_combined = df_combined.reset_index()

for col in df_combined.columns:
    if col != "date":
        df_combined[col] = df_combined[col].map({True: 1, False: 0}).fillna(0).astype(int)

df_combined["date"] = df_combined["date"].dt.strftime("%Y-%m-%d")
df_combined = df_combined.sort_values("date").reset_index(drop=True)

data = df_combined.to_dict(orient="records")

VALID_SIDES_FILTER = {"front", "right", "left"}
MIN_DAYS_PRESENT = 3
MIN_DAYS_ABSENT = 3

FACTORS = [col for col in df_combined.columns if col != "date"]

analysis_df = df.copy()
analysis_df = analysis_df[analysis_df["side"].str.lower().isin(VALID_SIDES_FILTER)].copy()

analysis_df["date"] = pd.to_datetime(analysis_df["date"], errors="coerce")
analysis_df["score"] = pd.to_numeric(analysis_df["score"], errors="coerce")
analysis_df = analysis_df.dropna(subset=["date", "score"]).copy()
analysis_df["date"] = analysis_df["date"].dt.normalize()
analysis_df = analysis_df.sort_values(["date", "side"]).reset_index(drop=True)

factors_df = df_combined.copy()
factors_df["date"] = pd.to_datetime(factors_df["date"], errors="coerce").dt.normalize()

for factor in FACTORS:
    if factor not in factors_df.columns:
        factors_df[factor] = 0
    factors_df[factor] = pd.to_numeric(factors_df[factor], errors="coerce").fillna(0).astype(int)

analysis_df = analysis_df.merge(factors_df[["date"] + FACTORS], on="date", how="left")
for factor in FACTORS:
    analysis_df[factor] = analysis_df[factor].fillna(0).astype(int)

daily_df = (
    analysis_df.groupby("date")
    .agg(
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        std_score=("score", "std"),
        total_acne_count=("acne_count", "sum"),
        n_images=("score", "count"),
    )
    .reset_index()
    .sort_values("date")
)

daily_df.head()

## WYKRESY

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

analysis_df["date"] = pd.to_datetime(analysis_df["date"])
daily_df["date"] = pd.to_datetime(daily_df["date"])

side_styles = {
    "left": {"color": "#E74C3C", "label": "Lewa strona"},
    "right": {"color": "#3498DB", "label": "Prawa strona"},
    "front": {"color": "#2ECC71", "label": "Przód"},
}

def plot_aggregated_scores(n_days=7, title=None, date_format="%Y-%m-%d"):
    if n_days < 1:
        raise ValueError("n_days musi być większe lub równe 1")

    title = title or f"Ewolucja zmian trądzikowych — agregacja co {n_days} dni"

    start_date = min(analysis_df["date"].min(), daily_df["date"].min()).normalize()

    def aggregate_by_days(df, value_col):
        temp = df[["date", value_col]].dropna().copy()
        temp["period"] = ((temp["date"].dt.normalize() - start_date).dt.days // n_days)
        return temp.groupby("period").agg(date=("date", "min"), score=(value_col, "mean")).set_index("date")["score"]

    fig, ax = plt.subplots(figsize=(12, 5), dpi=600)

    for side, style in side_styles.items():
        side_data = analysis_df[analysis_df["side"].str.lower() == side]
        if not side_data.empty:
            aggregated = aggregate_by_days(side_data, "score")
            if not aggregated.empty:
                ax.plot(aggregated.index, aggregated.values, marker="o", markersize=5, color=style["color"], linewidth=1.5, alpha=0.6, label=style["label"])

    overall = aggregate_by_days(daily_df, "mean_score")
    if not overall.empty:
        ax.plot(overall.index, overall.values, marker="o", markersize=6, color="#2C3E50", linewidth=2.5, label="Średnia ogólna", zorder=5)
        y_min, y_max = overall.min(), overall.max()
        margin = max((y_max - y_min) * 0.3, 0.1)
        ax.set_ylim(max(0, y_min - margin), min(1.0, y_max + margin))

    ax.set_xlabel("Data", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_ylabel("Średni Score", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=15, color="#2C3E50")
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter(date_format))
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, linestyle=":", alpha=0.6, color="#bdc3c7")
    ax.legend(frameon=True, facecolor="white", edgecolor="#bdc3c7", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_aggregated_scores(n_days=1)

plot_aggregated_scores(n_days=3)

plot_aggregated_scores(n_days=7)

plot_aggregated_scores(n_days=14)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt

factors_df = pd.DataFrame(data).copy()
factors_df["date"] = pd.to_datetime(factors_df["date"], errors="coerce").dt.normalize()

analysis_df["date"] = pd.to_datetime(analysis_df["date"], errors="coerce").dt.normalize()

daily_scores_df = (
    analysis_df.groupby("date")
    .agg(
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        std_score=("score", "std"),
        total_acne_count=("acne_count", "sum"),
        n_images=("score", "count"),
    )
    .reset_index()
    .sort_values("date")
)

daily_factor_df = pd.merge(factors_df, daily_scores_df, on="date", how="outer").sort_values("date")

for factor in FACTORS:
    if factor in daily_factor_df.columns:
        daily_factor_df[factor] = daily_factor_df[factor].fillna(0).astype(int)

daily_factor_df = daily_factor_df.sort_values("date").reset_index(drop=True)
daily_factor_df["interpolated_score"] = daily_factor_df["mean_score"].interpolate(method="linear").bfill().ffill()

min_date = daily_factor_df["date"].min()
max_date = daily_factor_df["date"].max()

def update_interactive_plot(selected_factor, show_present, start_date, end_date, n_days):
    if start_date > end_date:
        print("Data początkowa musi być wcześniejsza lub równa końcowej.")
        return

    df = daily_factor_df[
        (daily_factor_df["date"] >= pd.Timestamp(start_date)) &
        (daily_factor_df["date"] <= pd.Timestamp(end_date))
    ].copy()

    if df.empty:
        print("Brak danych w wybranym zakresie dat.")
        return

    n_days = max(1, int(n_days))
    start = pd.Timestamp(start_date)

    df["period"] = ((df["date"] - start).dt.days // n_days)

    aggregated = df.groupby("period").agg(
        date=("date", "min"),
        mean_score=("mean_score", "mean"),
        interpolated_score=("interpolated_score", "mean"),
        **{factor: (factor, "max" if show_present else "min") for factor in FACTORS if factor in df.columns}
    ).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 5), dpi=300)

    valid_scores = aggregated.dropna(subset=["mean_score"])
    ax.plot(valid_scores["date"], valid_scores["mean_score"], marker="o", markersize=5, color="#2C3E50", linewidth=2, label="Overall Mean Score", zorder=5)

    if selected_factor in aggregated.columns:
        condition_val = 1 if show_present else 0
        factor_subset = aggregated[aggregated[selected_factor] == condition_val]

        if not factor_subset.empty:
            status_label = "Present (1)" if show_present else "Absent (0)"
            ax.scatter(factor_subset["date"], factor_subset["interpolated_score"], color="#E74C3C", s=130, zorder=6, label=f"Factor '{selected_factor}' = {status_label}")

    all_scores = aggregated["interpolated_score"].dropna()
    if not all_scores.empty:
        y_min, y_max = all_scores.min(), all_scores.max()
        margin = max((y_max - y_min) * 0.25, 0.05)
        ax.set_ylim(max(0, y_min - margin), min(1.0, y_max + margin))

    ax.set_xlabel("Date", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_ylabel("Mean Score", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_title(f"Acne Score Evolution & Factor Impact: {selected_factor} ({n_days}-day groups)", fontsize=13, fontweight="bold", pad=15, color="#2C3E50")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, linestyle=":", alpha=0.6, color="#bdc3c7")
    ax.legend(frameon=True, facecolor="white", edgecolor="#bdc3c7", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

    plt.tight_layout()
    plt.show()

factor_dropdown = widgets.Dropdown(options=FACTORS, value=FACTORS[0], description="Factor:", style={"description_width": "initial"})
presence_checkbox = widgets.Checkbox(value=True, description="Present (1) / Absent (0)", style={"description_width": "initial"})
start_picker = widgets.DatePicker(description="Od:", value=min_date.date() if pd.notna(min_date) else None)
end_picker = widgets.DatePicker(description="Do:", value=max_date.date() if pd.notna(max_date) else None)
n_days_slider = widgets.IntSlider(value=7, min=1, max=30, step=1, description="Grupuj co N dni:", style={"description_width": "initial"})

widgets.interact(
    update_interactive_plot,
    selected_factor=factor_dropdown,
    show_present=presence_checkbox,
    start_date=start_picker,
    end_date=end_picker,
    n_days=n_days_slider
)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

factors = [c for c in factors_df.columns if c != "date"]
factors_ddf = factors_df.copy()
scores_df = daily_df[["date", "mean_score"]].copy()

factors_ddf["date"] = pd.to_datetime(factors_ddf["date"])
scores_df["date"] = pd.to_datetime(scores_df["date"])

df = factors_ddf.merge(scores_df, on="date", how="inner")
df = df.dropna(subset=["mean_score"])

results = []
for factor in factors:
    temp = df[[factor, "mean_score"]].dropna()
    if temp[factor].nunique() < 2 or len(temp) < 5:
        continue
    rho, p = spearmanr(temp[factor], temp["mean_score"])
    results.append({"czynnik": factor, "rho": rho, "p_value": p, "n": len(temp)})

corr_df = pd.DataFrame(results)
if not corr_df.empty:
    corr_df["p_adj"] = multipletests(corr_df["p_value"], method="fdr_bh")[1]
    corr_df["kierunek"] = np.where(corr_df["rho"] > 0, "Wyższy score", "Niższy score")
    corr_df = corr_df.sort_values("rho", ascending=False)

    display(corr_df.round(3))

    plot_df = corr_df.sort_values("rho")
    plt.figure(figsize=(10, max(4, len(plot_df) * 0.4)), dpi=150)
    colors = ["#E74C3C" if x > 0 else "#3498DB" for x in plot_df["rho"]]
    plt.barh(plot_df["czynnik"], plot_df["rho"], color=colors)
    plt.axvline(0, color="black", linewidth=0.8)
    plt.xlabel("Korelacja Spearmana z mean_score")
    plt.title("Korelacja czynników z acne score")
    plt.grid(axis="x", linestyle=":", alpha=0.4)
    plt.tight_layout()
    plt.show()
else:
    print("Brak czynników z wystarczającą zmiennością do obliczenia korelacji.")

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests

MAX_LAG = 14
MIN_N = 10

factors_df = factors_df.copy()
scores_df = daily_df[["date", "mean_score"]].copy()

factors_df["date"] = pd.to_datetime(factors_df["date"]).dt.normalize()
scores_df["date"] = pd.to_datetime(scores_df["date"]).dt.normalize()

factors = [c for c in factors_df.columns if c != "date"]

factors_df = factors_df.drop_duplicates("date").set_index("date")
scores_df = scores_df.drop_duplicates("date").set_index("date")

calendar = pd.date_range(
    min(factors_df.index.min(), scores_df.index.min()),
    max(factors_df.index.max(), scores_df.index.max()),
    freq="D"
)

data = factors_df.reindex(calendar).join(
    scores_df.reindex(calendar), how="left"
)
data.index.name = "date"

for factor in factors:
    data[factor] = pd.to_numeric(data[factor], errors="coerce")

results = []

for factor in factors:
    for lag in range(MAX_LAG + 1):
        temp = pd.DataFrame({
            "factor": data[factor],
            "score_future": data["mean_score"].shift(-lag)
        }).dropna()

        if len(temp) < MIN_N or temp["factor"].nunique() < 2:
            continue

        rho, p = spearmanr(temp["factor"], temp["score_future"])
        present = temp.loc[temp["factor"] == 1, "score_future"]
        absent = temp.loc[temp["factor"] == 0, "score_future"]

        results.append({
            "czynnik": factor,
            "lag_dni": lag,
            "rho": rho,
            "p_value": p,
            "n": len(temp),
            "n_obecny": len(present),
            "n_nieobecny": len(absent),
            "score_przy_obecnosci": present.mean(),
            "score_bez_czynnika": absent.mean(),
            "roznica_score": present.mean() - absent.mean()
        })

lag_df = pd.DataFrame(results)

if lag_df.empty:
    print("Za mało kompletnych danych do analizy. Sprawdź log i liczbę dni.")
else:
    lag_df["p_adj"] = multipletests(
        lag_df["p_value"], method="fdr_bh"
    )[1]

    # Najsilniejsza zależność dla każdego czynnika
    best_idx = lag_df.groupby("czynnik")["rho"].apply(
        lambda s: s.abs().idxmax()
    )
    best = lag_df.loc[best_idx].copy()
    best["kierunek"] = np.where(
        best["rho"] > 0, "Wyższy score", "Niższy score"
    )
    best = best.sort_values("rho", key=lambda s: s.abs(), ascending=False)

    display(best[[
        "czynnik", "lag_dni", "rho", "roznica_score",
        "p_value", "p_adj", "n"
    ]].round(3))

    # Wykres: korelacja każdego czynnika przy każdym opóźnieniu
    heat = lag_df.pivot(index="czynnik", columns="lag_dni", values="rho")
    plt.figure(figsize=(13, max(4, len(heat) * 0.55)), dpi=150)
    sns.heatmap(
        heat, annot=True, fmt=".2f", center=0,
        cmap="RdBu_r", vmin=-1, vmax=1,
        linewidths=0.3
    )
    plt.title("Korelacja czynnika ze score'em po określonej liczbie dni")
    plt.xlabel("Opóźnienie (dni)")
    plt.ylabel("Czynnik")
    plt.tight_layout()
    plt.show()

    # Wykres różnicy średniego score'u przy obecności i braku czynnika
    plt.figure(figsize=(12, 5), dpi=150)
    for factor in factors:
        subset = lag_df[lag_df["czynnik"] == factor]
        if not subset.empty:
            plt.plot(
                subset["lag_dni"], subset["roznica_score"],
                marker="o", markersize=3, alpha=0.7, label=factor
            )
    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Opóźnienie (dni)")
    plt.ylabel("Różnica średniego score'u (obecny - nieobecny)")
    plt.title("Jak zmienia się zależność wraz z opóźnieniem")
    plt.grid(linestyle=":", alpha=0.4)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()


In [ ]:
lag_df["p_adj"] = multipletests(lag_df["p_value"], method="fdr_bh")[1]

SIGNIFICANCE_LEVEL = 0.05
significant = lag_df[lag_df["p_adj"] < SIGNIFICANCE_LEVEL].copy()

if significant.empty:
    print(f"Brak wyników z p_adj < {SIGNIFICANCE_LEVEL}.")
else:
    best_idx = significant.groupby("czynnik")["rho"].apply(
        lambda s: s.abs().idxmax()
    )
    best = significant.loc[best_idx].copy()
    best["kierunek"] = np.where(best["rho"] > 0, "Wyższy score", "Niższy score")
    best = best.sort_values("rho", key=lambda s: s.abs(), ascending=False)

    display(best[[
        "czynnik", "lag_dni", "rho", "roznica_score",
        "p_value", "p_adj", "n"
    ]].round(3))

    selected_factors = best["czynnik"].unique()
    heat = significant[significant["czynnik"].isin(selected_factors)].pivot(
        index="czynnik", columns="lag_dni", values="rho"
    )

    plt.figure(figsize=(13, max(3, len(heat) * 0.55)), dpi=150)
    sns.heatmap(heat, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
                vmin=-1, vmax=1, linewidths=0.3)
    plt.title("Istotne korelacje z acne score według opóźnienia")
    plt.xlabel("Opóźnienie (dni)")
    plt.ylabel("Czynnik")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5), dpi=150)
    for factor in selected_factors:
        subset = significant[significant["czynnik"] == factor].sort_values("lag_dni")
        plt.plot(subset["lag_dni"], subset["roznica_score"], marker="o",
                 markersize=3, alpha=0.8, label=factor)

    plt.axhline(0, color="black", linewidth=0.8)
    plt.xlabel("Opóźnienie (dni)")
    plt.ylabel("Różnica średniego score'u (obecny − nieobecny)")
    plt.title("Różnica acne score dla istotnych zależności")
    plt.grid(linestyle=":", alpha=0.4)
    plt.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    plt.show()

In [ ]:
if "poczatek_okresu" not in daily_factor_df.columns:
    print("Brak danych: kolumna 'poczatek_okresu' nie istnieje w danych.")
else:
    cycle_df = daily_factor_df.copy()
    cycle_df = cycle_df.sort_values("date").reset_index(drop=True)

    cycle_starts = cycle_df.loc[cycle_df["poczatek_cyklu"] == 1, "date"].tolist()
    if not cycle_starts and not cycle_df.empty:
        cycle_starts = [cycle_df["date"].min()]

    cycle_days = []
    for d in cycle_df["date"]:
        valid_starts = [s for s in cycle_starts if s <= d]
        if valid_starts:
            latest_start = max(valid_starts)
            diff_days = (d - latest_start).days
            cycle_day = (diff_days % 28) + 1
        else:
            first_start = cycle_starts[0]
            diff_days = (first_start - d).days
            cycle_day = 28 - (diff_days % 28)
        cycle_days.append(cycle_day)

    cycle_df["cycle_day"] = cycle_days

    def get_cycle_phase(day):
        if pd.isna(day):
            return "Brak danych"
        elif 1 <= day <= 5:
            return "Menstruacja (Dni 1-5)"
        elif 6 <= day <= 13:
            return "Faza folikularna (Dni 6-13)"
        elif 14 <= day <= 16:
            return "Owulacja (Dni 14-16)"
        else:
            return "Faza lutealna (Dni 17-28)"

    cycle_df["cycle_phase"] = cycle_df["cycle_day"].apply(get_cycle_phase)


    phase_colors = {
        "Menstruacja (Dni 1-5)": "#FF9999", 
        "Faza folikularna (Dni 6-13)": "#85C1E9",
        "Owulacja (Dni 14-16)": "#7DCEA0",  
        "Faza lutealna (Dni 17-28)": "#BB8FCE", 
    }

    print("\n=== PODSUMOWANIE DNI I FAZ CYKLU ===")
    display(cycle_df[["date", "cycle_day", "cycle_phase", "mean_score"]].dropna(subset=["mean_score"]))


    fig, ax = plt.subplots(figsize=(13, 5.5), dpi=300)

    valid_scores = cycle_df.dropna(subset=["mean_score"])

    if not cycle_df.empty:
        min_date = cycle_df["date"].min()
        max_date = cycle_df["date"].max()
        
        current_phase = None
        span_start = None
        
        temp_df = cycle_df.sort_values("date").copy()
        
        for i, row in temp_df.iterrows():
            p = row["cycle_phase"]
            dt = row["date"]
            
            if p != current_phase:
                if current_phase is not None and current_phase in phase_colors:
                    ax.axvspan(span_start, dt, color=phase_colors[current_phase], alpha=0.55, lw=0, label=current_phase)
                current_phase = p
                span_start = dt
                
        if current_phase is not None and current_phase in phase_colors:
            ax.axvspan(span_start, max_date + pd.Timedelta(days=1), color=phase_colors[current_phase], alpha=0.25, lw=0, label=current_phase)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ordered_labels = [l for l in phase_colors.keys() if l in by_label]
    ordered_handles = [by_label[l] for l in ordered_labels]

    ax.plot(valid_scores["date"], valid_scores["mean_score"], marker="o", markersize=6, color="#2C3E50", linewidth=2.5, label="Średni Score", zorder=5)

    ordered_labels.insert(0, "Średni Score")
    ordered_handles.insert(0, ax.get_lines()[0])

    if not valid_scores["mean_score"].empty:
        y_min = valid_scores["mean_score"].min()
        y_max = valid_scores["mean_score"].max()
        margin = max((y_max - y_min) * 0.3, 0.08)
        ax.set_ylim(max(0, y_min - margin), min(1.0, y_max + margin))

    ax.set_xlabel("Data", fontsize=11, fontweight='bold', color="#2C3E50")
    ax.set_ylabel("Średni Score", fontsize=11, fontweight='bold', color="#2C3E50")
    ax.set_title("Ewolucja Acne Score w czasie z podziałem na sfery faz cyklu", fontsize=13, fontweight='bold', pad=15, color="#2C3E50")
    ax.tick_params(axis='x', rotation=30)
    ax.grid(True, linestyle=":", alpha=0.6, color="#bdc3c7")

    ax.legend(ordered_handles, ordered_labels, frameon=True, facecolor='white', edgecolor='#bdc3c7', fontsize=9, loc='upper left')

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
from pathlib import Path
import sys
import gc

import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# PROJECT PATHS

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predict import load_model



DATASET_DIR = PROJECT_ROOT / "images" / "zuza" / "dated_fotoes"
MAP_OUTPUT_DIR = BASE_DIR / "acne_maps"
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = BASE_DIR / "face_landmarker.task"

# PROCESSING SETTINGS

TARGET_SIDE = "front"  # Options: "right", "left", "front"

LEFT_THRESHOLD = 0.30
RIGHT_THRESHOLD = 0.70

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

MAP_WIDTH = 600
MAP_HEIGHT = 800

FACE_ALPHA = 0.055
HEATMAP_SIGMA = 12
HEATMAP_ALPHA = 0.78
YOLO_CONFIDENCE = 0.25

SIDE_NAME = TARGET_SIDE.lower()

# INITIALIZE MEDIAPIPE

print("MediaPipe:", mp.__version__)

base_options = python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)

face_landmarker = vision.FaceLandmarker.create_from_options(options)
print("Face Landmarker: OK")


# FACE DETECTION AND ALIGNMENT

import json
from pathlib import Path


USE_CACHED_PREDICTIONS = True

PREDICTIONS_ROOT = Path(
    PROJECT_ROOT / "runs" / "batch"
)




def classify_profile(landmarks):
    """Classify face orientation using the nose position."""
    nose_x = landmarks[1].x
    left_x = landmarks[234].x
    right_x = landmarks[454].x
    face_width = right_x - left_x

    if face_width == 0:
        return "unclassified", 0.0

    ratio = (nose_x - left_x) / face_width

    if ratio < LEFT_THRESHOLD:
        side = "left"
    elif ratio > RIGHT_THRESHOLD:
        side = "right"
    else:
        side = "front"

    return side, ratio


def detect_landmarks(image_bgr):
    """Detect face landmarks in a BGR image."""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = face_landmarker.detect(mp_image)
    return result.face_landmarks[0] if result.face_landmarks else None


def landmarks_to_pixels(landmarks, width, height):
    """Convert normalized landmarks to pixel coordinates."""
    return np.asarray([[lm.x * width, lm.y * height] for lm in landmarks], dtype=np.float32)


def find_target_images(dataset_dir, target_side):
    """Find images stored in folders matching the target side."""
    images = []

    for path in dataset_dir.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        parts = path.relative_to(dataset_dir).parts
        if len(parts) >= 3 and parts[1].lower() == target_side:
            images.append(path)

    return sorted(images)


def scale_points_to_output(points, src_width, src_height):
    """Scale pixel coordinates to the output map dimensions."""
    points = points.copy()
    points[:, 0] *= MAP_WIDTH / src_width
    points[:, 1] *= MAP_HEIGHT / src_height
    return points


def estimate_alignment_transform(
    source_landmarks, reference_landmarks,
    source_width, source_height,
    reference_width, reference_height
):
    """Estimate an affine transform between two sets of face landmarks."""
    source = landmarks_to_pixels(source_landmarks, source_width, source_height)
    reference = landmarks_to_pixels(reference_landmarks, reference_width, reference_height)
    reference = scale_points_to_output(reference, reference_width, reference_height)

    if len(source) != 478 or len(reference) != 478:
        raise ValueError("Expected 478 landmarks.")

    matrix, _ = cv2.estimateAffinePartial2D(
        source, reference,
        method=cv2.RANSAC,
        ransacReprojThreshold=3.0,
        maxIters=5000,
        confidence=0.99,
        refineIters=50,
    )

    if matrix is None:
        raise RuntimeError("Could not estimate face alignment.")

    return matrix


def transform_points(points, matrix):
    """Apply an affine transform to a set of points."""
    if len(points) == 0:
        return np.empty((0, 2), dtype=np.float32)

    ones = np.ones((len(points), 1), dtype=np.float32)
    return (np.hstack((points, ones)) @ matrix.T).astype(np.float32)


def align_image(image, landmarks, reference_landmarks, reference_width, reference_height):
    """Align a face image to the reference dimensions and landmarks."""
    height, width = image.shape[:2]

    matrix = estimate_alignment_transform(
        landmarks, reference_landmarks,
        width, height,
        reference_width, reference_height,
    )

    aligned = cv2.warpAffine(
        image, matrix,
        (MAP_WIDTH, MAP_HEIGHT),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT,
    )

    return aligned, matrix


# ACNE DETECTION AND HEATMAP

def load_cached_detections(image_path):
    """Load cached detections for an image, if available."""
    image_path = Path(image_path)

    if not PREDICTIONS_ROOT.exists():
        return None

    candidates = sorted(
        PREDICTIONS_ROOT.glob(
            f"*/img_{image_path.stem}/results.json"
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    for json_path in candidates:
        try:
            with json_path.open("r", encoding="utf-8") as file:
                data = json.load(file)

            cached_image = data.get("image", {}).get("input")

            if cached_image and Path(cached_image).name != image_path.name:
                continue


            detections = data.get("detections")

            if not isinstance(detections, list):
                continue

            print(f"  -> cached predictions: {json_path}")
            return detections

        except (OSError, json.JSONDecodeError, TypeError, ValueError) as error:
            print(f"  -> invalid cache {json_path}: {error}")

    return None



# ============================================================
# DOZWOLONE KLASY I ICH WAGI NA HEATMAPIE
# ============================================================
# ALLOWED_CLASSES = {
#     "acne": 1.0,           # model v3 / ogólne
#     "closed_comedo": 0.7,  # zaskórnik zamknięty
#     "open_comedo": 0.8,    # zaskórnik otwarty
#     "papule": 1.0,         # grudka
#     "pustule": 1.2,        # krosta
#     "nodule": 1.5,         # guzek trądzikowy
# }

ALLOWED_CLASSES = {
    "closed_comedo": 0.3,  # Zaskórnik zamknięty (bardzo łagodny, stonowany)
    "open_comedo": 0.4,    # Zaskórnik otwarty
    "acne": 1.0,           # Standardowy model v3 (baza referencyjna)
    "papule": 1.1,         # Grudka (lekkie/średnie zapalenie)
    "pustule": 1.6,        # Krosta (wyraźny stan zapalny)
    "nodule": 2.5,         # Guzek trądzikowy (ciężki, głęboki stan zapalny — mocno wyróżniony)
}

def detect_acne(image_bgr, image_path):
    """Return acne boxes in xyxy format with class-weighted confidence scores."""
    detections = None

    if USE_CACHED_PREDICTIONS:
        detections = load_cached_detections(image_path)

    if detections is not None:
        output = []

        for detection in detections:
            class_name = str(detection.get("class", "acne")).lower()

            # Odrzucamy klasy spoza słownika dozwolonych zmian trądzikowych
            if class_name not in ALLOWED_CLASSES:
                continue

            class_weight = ALLOWED_CLASSES[class_name]
            bbox = detection.get("bbox")

            if bbox is None or len(bbox) != 4:
                continue

            # Konwersja YOLO xywh -> xyxy
            cx, cy, bw, bh = map(float, bbox)

            x1 = cx - bw / 2
            y1 = cy - bh / 2
            x2 = cx + bw / 2
            y2 = cy + bh / 2

            box = np.array([x1, y1, x2, y2], dtype=np.float32)
            confidence = float(detection.get("confidence", 0.0))

            # Łączymy pewność detekcji z wagą ciężkości danej klasy
            effective_confidence = confidence * class_weight

            output.append((box, effective_confidence))

        return output

    # Fallback: bezpośrednia predykcja YOLO, gdy brak skanów w keszu
    print("  -> no matching cache; running YOLO prediction")

    results = yolo_model.predict(
        source=image_bgr,
        conf=YOLO_CONFIDENCE,
        verbose=False,
    )

    if not results or results[0].boxes is None:
        return []

    boxes = results[0].boxes
    xyxy = boxes.xyxy.cpu().numpy()

    confidences = (
        boxes.conf.cpu().numpy()
        if boxes.conf is not None
        else np.ones(len(xyxy), dtype=np.float32)
    )

    output = []
    for box, confidence in zip(xyxy, confidences):
        cls_id = int(box.cls[0]) if hasattr(box, "cls") else 0
        class_name = str(yolo_model.names.get(cls_id, "acne")).lower()

        if class_name not in ALLOWED_CLASSES:
            continue

        class_weight = ALLOWED_CLASSES[class_name]
        effective_confidence = float(confidence) * class_weight
        output.append((box.astype(np.float32), effective_confidence))

    return output

def add_heat_point(heatmap, x, y, weight, sigma):
    """Add a weighted Gaussian point to the heatmap."""
    height, width = heatmap.shape
    radius = int(sigma * 4)

    x0, x1 = max(0, int(x - radius)), min(width, int(x + radius + 1))
    y0, y1 = max(0, int(y - radius)), min(height, int(y + radius + 1))

    if x0 >= x1 or y0 >= y1:
        return

    yy, xx = np.mgrid[y0:y1, x0:x1]
    gaussian = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    heatmap[y0:y1, x0:x1] += gaussian * weight


# FIND TARGET IMAGES

target_images = find_target_images(DATASET_DIR, TARGET_SIDE)

print()
print(f"{TARGET_SIDE.upper()} images found:", len(target_images))

if not target_images:
    raise RuntimeError(f"Nie znaleziono zdjęć w folderach /{TARGET_SIDE}/.")


# LOAD YOLO MODEL

yolo_model = load_model()
print("YOLO: OK")


# FIND REFERENCE IMAGE

reference_image = None
reference_landmarks = None
reference_image_path = None

print()
print(f"Searching for reference {TARGET_SIDE.upper()} face...")

for path in target_images:
    image = cv2.imread(str(path))

    if image is None:
        continue

    landmarks = detect_landmarks(image)

    if landmarks is None:
        del image
        continue

    orientation, _ = classify_profile(landmarks)

    if orientation == TARGET_SIDE:
        reference_image = image
        reference_landmarks = landmarks
        reference_image_path = path
        break

    del image, landmarks

if reference_landmarks is None:
    raise RuntimeError(f"Nie znaleziono poprawnego zdjęcia {TARGET_SIDE.upper()}.")

reference_height, reference_width = reference_image.shape[:2]

print("Reference:", reference_image_path)
print("Reference size:", reference_width, "x", reference_height)

del reference_image
gc.collect()


# INITIALIZE IMAGE ACCUMULATOR AND HEATMAP

image_accumulator = np.zeros((MAP_HEIGHT, MAP_WIDTH, 3), dtype=np.float32)
heatmap = np.zeros((MAP_HEIGHT, MAP_WIDTH), dtype=np.float32)

successful_images = 0
total_detections = 0


# PROCESS TARGET IMAGES

print()
print("=" * 70)
print(f"PROCESSING {TARGET_SIDE.upper()} PROFILE IMAGES")
print("=" * 70)

for index, image_path in enumerate(target_images, 1):
    print(f"[{index}/{len(target_images)}] {image_path.name}")

    image = cv2.imread(str(image_path))

    if image is None:
        print("  -> skipped: cannot read image")
        continue

    landmarks = detect_landmarks(image)

    if landmarks is None:
        print("  -> skipped: no face")
        del image
        continue

    orientation, nose_ratio = classify_profile(landmarks)

    print(f"  orientation = {orientation}")
    print(f"  nose ratio  = {nose_ratio:.4f}")

    if orientation != TARGET_SIDE:
        print(f"  -> skipped: not {TARGET_SIDE.upper()}")
        del image, landmarks
        continue

    try:
        aligned_image, matrix = align_image(
            image, landmarks, reference_landmarks,
            reference_width, reference_height,
        )
    except Exception as e:
        print("  -> skipped: alignment failed:", e)
        del image, landmarks
        continue

    # Detect acne on the original image.
    detections = detect_acne(image, image_path)

    print(f"  acne detections = {len(detections)}")
    total_detections += len(detections)

    # Accumulate aligned face images.
    image_accumulator += aligned_image.astype(np.float32) * FACE_ALPHA

    # Project acne centers onto the aligned heatmap.
    for box, confidence in detections:
        x1, y1, x2, y2 = box
        center = np.array([[(x1 + x2) * 0.5, (y1 + y2) * 0.5]], dtype=np.float32)
        aligned_center = transform_points(center, matrix)[0]
        x, y = aligned_center

        if 0 <= x < MAP_WIDTH and 0 <= y < MAP_HEIGHT:
            # add_heat_point(heatmap, x, y, confidence, HEATMAP_SIGMA)
            weight = confidence ** 3
            add_heat_point(heatmap, x, y, weight, HEATMAP_SIGMA)

    successful_images += 1

    del image, landmarks, aligned_image, matrix, detections

    if index % 10 == 0:
        gc.collect()


# PRINT PROCESSING SUMMARY

print()
print("=" * 70)
print("PROCESSING SUMMARY")
print("=" * 70)

print(f"Images found:        {len(target_images)}")
print(f"Images aligned:      {successful_images}")
print(f"Total acne lesions:  {total_detections}")

if successful_images:
    print(f"Mean lesions/image:  {total_detections / successful_images:.2f}")

if successful_images == 0:
    raise RuntimeError(f"Nie udało się wyrównać żadnego zdjęcia {TARGET_SIDE.upper()}.")


# CREATE AVERAGED FACE STACK

face_stack = image_accumulator / (FACE_ALPHA * successful_images)
face_stack = np.clip(face_stack, 0, 255).astype(np.uint8)

del image_accumulator
gc.collect()


# NORMALIZE HEATMAP INTENSITY

heatmap_mean = heatmap / successful_images
nonzero_values = heatmap_mean[heatmap_mean > 0]

if len(nonzero_values):
    heatmap_max = np.percentile(nonzero_values, 99)
else:
    heatmap_max = 1.0

heatmap_normalized = np.clip(heatmap_mean / max(heatmap_max, 1e-8), 0, 1)

del heatmap_mean, nonzero_values
gc.collect()


# APPLY HEATMAP COLORS

heatmap_uint8 = (heatmap_normalized * 255).astype(np.uint8)
heatmap_color_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

del heatmap_uint8
gc.collect()


# OVERLAY HEATMAP ON FACE STACK

heat_alpha = (heatmap_normalized * HEATMAP_ALPHA)[..., None]
final_map = (
    face_stack.astype(np.float32) * (1.0 - heat_alpha)
    + heatmap_color_bgr.astype(np.float32) * heat_alpha
)

final_map = np.clip(final_map, 0, 255).astype(np.uint8)

del heat_alpha
gc.collect()


# SAVE OUTPUT FILES

heatmap_npy_path = MAP_OUTPUT_DIR / f"{SIDE_NAME}_acne_intensity.npy"
face_stack_path = MAP_OUTPUT_DIR / f"{SIDE_NAME}_aligned_face_stack.png"
heatmap_path = MAP_OUTPUT_DIR / f"{SIDE_NAME}_acne_heatmap.png"
final_map_path = MAP_OUTPUT_DIR / f"{SIDE_NAME}_acne_map.png"

np.save(heatmap_npy_path, heatmap_normalized)
cv2.imwrite(str(face_stack_path), face_stack)
cv2.imwrite(str(heatmap_path), heatmap_color_bgr)
cv2.imwrite(str(final_map_path), final_map)

print()
print("=" * 70)
print("OUTPUT")
print("=" * 70)

print(face_stack_path)
print(heatmap_path)
print(final_map_path)
print(heatmap_npy_path)


# DISPLAY RESULTS

plt.figure(figsize=(7, 9))
plt.imshow(cv2.cvtColor(face_stack, cv2.COLOR_BGR2RGB))
plt.title(f"{TARGET_SIDE.upper()} — full aligned image stack\nn = {successful_images}")
plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 9))
plt.imshow(heatmap_normalized, cmap="jet", vmin=0, vmax=1)
plt.colorbar(label="Relative acne intensity")
plt.title(f"{TARGET_SIDE.upper()} — acne intensity")
plt.axis("off")
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 9))
plt.imshow(cv2.cvtColor(final_map, cv2.COLOR_BGR2RGB))
plt.title(
    f"{TARGET_SIDE.upper()} — Acne Map\n"
    f"{successful_images} aligned faces / {total_detections} detections"
)
plt.axis("off")
plt.tight_layout()
plt.show()


# RELEASE UNUSED MEMORY

del face_stack, heatmap, heatmap_normalized, heatmap_color_bgr, final_map, target_images
gc.collect()

In [ ]:
from pathlib import Path
import sys
import gc

import cv2
import numpy as np
import matplotlib.pyplot as plt
import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# PROJECT PATHS

PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predict import load_model


DATASET_DIR = DATASET_DIR
MAP_OUTPUT_DIR = BASE_DIR / "acne_maps"
MAP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = BASE_DIR / "face_landmarker.task"

# PROCESSING SETTINGS

TARGET_SIDE = "front"  # Options: "right", "left", "front"

LEFT_THRESHOLD = 0.30
RIGHT_THRESHOLD = 0.70

VALID_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}

MAP_WIDTH = 600
MAP_HEIGHT = 800

FACE_ALPHA = 0.055
HEATMAP_SIGMA = 12
HEATMAP_ALPHA = 0.78
YOLO_CONFIDENCE = 0.25

SIDE_NAME = TARGET_SIDE.lower()

# INITIALIZE MEDIAPIPE

print("MediaPipe:", mp.__version__)

base_options = python.BaseOptions(model_asset_path=str(MODEL_PATH))
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    running_mode=vision.RunningMode.IMAGE,
    num_faces=1,
    output_face_blendshapes=False,
    output_facial_transformation_matrixes=False,
)

face_landmarker = vision.FaceLandmarker.create_from_options(options)
print("Face Landmarker: OK")


# FACE DETECTION AND ALIGNMENT

import json
from pathlib import Path

USE_CACHED_PREDICTIONS = True
PREDICTIONS_ROOT = Path(PROJECT_ROOT / "runs" / "batch")


def classify_profile(landmarks):
    """Classify face orientation using the nose position."""
    nose_x = landmarks[1].x
    left_x = landmarks[234].x
    right_x = landmarks[454].x
    face_width = right_x - left_x

    if face_width == 0:
        return "unclassified", 0.0

    ratio = (nose_x - left_x) / face_width

    if ratio < LEFT_THRESHOLD:
        side = "left"
    elif ratio > RIGHT_THRESHOLD:
        side = "right"
    else:
        side = "front"

    return side, ratio


def detect_landmarks(image_bgr):
    """Detect face landmarks in a BGR image."""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
    result = face_landmarker.detect(mp_image)
    return result.face_landmarks[0] if result.face_landmarks else None


def landmarks_to_pixels(landmarks, width, height):
    """Convert normalized landmarks to pixel coordinates."""
    return np.asarray([[lm.x * width, lm.y * height] for lm in landmarks], dtype=np.float32)


def find_target_images(dataset_dir, target_side):
    """Find images stored in folders matching the target side."""
    images = []

    for path in dataset_dir.rglob("*"):
        if not path.is_file() or path.suffix.lower() not in VALID_EXTENSIONS:
            continue

        parts = path.relative_to(dataset_dir).parts
        if len(parts) >= 3 and parts[1].lower() == target_side:
            images.append(path)

    return sorted(images)


def scale_points_to_output(points, src_width, src_height):
    """Scale pixel coordinates to the output map dimensions."""
    points = points.copy()
    points[:, 0] *= MAP_WIDTH / src_width
    points[:, 1] *= MAP_HEIGHT / src_height
    return points


def estimate_alignment_transform(
    source_landmarks, reference_landmarks,
    source_width, source_height,
    reference_width, reference_height
):
    """Estimate an affine transform between two sets of face landmarks."""
    source = landmarks_to_pixels(source_landmarks, source_width, source_height)
    reference = landmarks_to_pixels(reference_landmarks, reference_width, reference_height)
    reference = scale_points_to_output(reference, reference_width, reference_height)

    if len(source) != 478 or len(reference) != 478:
        raise ValueError("Expected 478 landmarks.")

    matrix, _ = cv2.estimateAffinePartial2D(
        source, reference,
        method=cv2.RANSAC,
        ransacReprojThreshold=3.0,
        maxIters=5000,
        confidence=0.99,
        refineIters=50,
    )

    if matrix is None:
        raise RuntimeError("Could not estimate face alignment.")

    return matrix


def transform_points(points, matrix):
    """Apply an affine transform to a set of points."""
    if len(points) == 0:
        return np.empty((0, 2), dtype=np.float32)

    ones = np.ones((len(points), 1), dtype=np.float32)
    return (np.hstack((points, ones)) @ matrix.T).astype(np.float32)


def align_image(image, landmarks, reference_landmarks, reference_width, reference_height):
    """Align a face image to the reference dimensions and landmarks."""
    height, width = image.shape[:2]

    matrix = estimate_alignment_transform(
        landmarks, reference_landmarks,
        width, height,
        reference_width, reference_height,
    )

    aligned = cv2.warpAffine(
        image, matrix,
        (MAP_WIDTH, MAP_HEIGHT),
        flags=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_REFLECT,
    )

    return aligned, matrix


# ACNE DETECTION AND HEATMAP

def load_cached_detections(image_path):
    """Load cached detections for an image, if available."""
    image_path = Path(image_path)

    if not PREDICTIONS_ROOT.exists():
        return None

    candidates = sorted(
        PREDICTIONS_ROOT.glob(
            f"*/img_{image_path.stem}/results.json"
        ),
        key=lambda path: path.stat().st_mtime,
        reverse=True,
    )

    for json_path in candidates:
        try:
            with json_path.open("r", encoding="utf-8") as file:
                data = json.load(file)

            cached_image = data.get("image", {}).get("input")

            if cached_image and Path(cached_image).name != image_path.name:
                continue

            detections = data.get("detections")

            if not isinstance(detections, list):
                continue

            print(f"  -> cached predictions: {json_path}")
            return detections

        except (OSError, json.JSONDecodeError, TypeError, ValueError) as error:
            print(f"  -> invalid cache {json_path}: {error}")

    return None


# ============================================================
# DEFINICJA KLAS I ETYKIET OPISOWYCH (PL)
# ============================================================
ALLOWED_CLASSES = {
    "closed_comedo": {"weight": 0.3, "pl": "Zaskórnik zamknięty"},
    "open_comedo": {"weight": 0.4, "pl": "Zaskórnik otwarty"},
    "acne": {"weight": 1.0, "pl": "Trądzik ogólny"},
    "papule": {"weight": 1.1, "pl": "Grudka"},
    "pustule": {"weight": 1.6, "pl": "Krosta"},
    "nodule": {"weight": 2.5, "pl": "Guzek trądzikowy"},
}

def detect_acne(image_bgr, image_path):
    """Return acne boxes grouped by class with class-weighted confidence scores."""
    detections = None

    if USE_CACHED_PREDICTIONS:
        detections = load_cached_detections(image_path)

    if detections is not None:
        output = []

        for detection in detections:
            class_name = str(detection.get("class", "acne")).lower()

            if class_name not in ALLOWED_CLASSES:
                continue

            class_info = ALLOWED_CLASSES[class_name]
            bbox = detection.get("bbox")

            if bbox is None or len(bbox) != 4:
                continue

            cx, cy, bw, bh = map(float, bbox)
            x1 = cx - bw / 2
            y1 = cy - bh / 2
            x2 = cx + bw / 2
            y2 = cy + bh / 2

            box = np.array([x1, y1, x2, y2], dtype=np.float32)
            confidence = float(detection.get("confidence", 0.0))
            effective_confidence = confidence * class_info["weight"]

            output.append((class_name, box, effective_confidence))

        return output

    # Fallback: bezpośrednia predykcja YOLO
    print("  -> no matching cache; running YOLO prediction")

    results = yolo_model.predict(
        source=image_bgr,
        conf=YOLO_CONFIDENCE,
        verbose=False,
    )

    if not results or results[0].boxes is None:
        return []

    boxes = results[0].boxes
    xyxy = boxes.xyxy.cpu().numpy()

    confidences = (
        boxes.conf.cpu().numpy()
        if boxes.conf is not None
        else np.ones(len(xyxy), dtype=np.float32)
    )

    output = []
    for box, confidence in zip(xyxy, confidences):
        cls_id = int(box.cls[0]) if hasattr(box, "cls") else 0
        class_name = str(yolo_model.names.get(cls_id, "acne")).lower()

        if class_name not in ALLOWED_CLASSES:
            continue

        class_info = ALLOWED_CLASSES[class_name]
        effective_confidence = float(confidence) * class_info["weight"]
        output.append((class_name, box.astype(np.float32), effective_confidence))

    return output


def add_heat_point(heatmap, x, y, weight, sigma):
    """Add a weighted Gaussian point to the heatmap."""
    height, width = heatmap.shape
    radius = int(sigma * 4)

    x0, x1 = max(0, int(x - radius)), min(width, int(x + radius + 1))
    y0, y1 = max(0, int(y - radius)), min(height, int(y + radius + 1))

    if x0 >= x1 or y0 >= y1:
        return

    yy, xx = np.mgrid[y0:y1, x0:x1]
    gaussian = np.exp(-((xx - x) ** 2 + (yy - y) ** 2) / (2 * sigma ** 2))
    heatmap[y0:y1, x0:x1] += gaussian * weight


# FIND TARGET IMAGES

target_images = find_target_images(DATASET_DIR, TARGET_SIDE)

print()
print(f"{TARGET_SIDE.upper()} images found:", len(target_images))

if not target_images:
    raise RuntimeError(f"Nie znaleziono zdjęć w folderach /{TARGET_SIDE}/.")


# LOAD YOLO MODEL

yolo_model = load_model()
print("YOLO: OK")


# FIND REFERENCE IMAGE

reference_image = None
reference_landmarks = None
reference_image_path = None

print()
print(f"Searching for reference {TARGET_SIDE.upper()} face...")

for path in target_images:
    image = cv2.imread(str(path))

    if image is None:
        continue

    landmarks = detect_landmarks(image)

    if landmarks is None:
        del image
        continue

    orientation, _ = classify_profile(landmarks)

    if orientation == TARGET_SIDE:
        reference_image = image
        reference_landmarks = landmarks
        reference_image_path = path
        break

    del image, landmarks

if reference_landmarks is None:
    raise RuntimeError(f"Nie znaleziono poprawnego zdjęcia {TARGET_SIDE.upper()}.")

reference_height, reference_width = reference_image.shape[:2]

print("Reference:", reference_image_path)
print("Reference size:", reference_width, "x", reference_height)

del reference_image
gc.collect()


# INITIALIZE IMAGE ACCUMULATOR AND CLASS HEATMAPS

image_accumulator = np.zeros((MAP_HEIGHT, MAP_WIDTH, 3), dtype=np.float32)

# Słownik przechowujący osobną macierz heatmapy oraz licznik detekcji dla każdej klasy
class_heatmaps = {cls_name: np.zeros((MAP_HEIGHT, MAP_WIDTH), dtype=np.float32) for cls_name in ALLOWED_CLASSES}
class_counts = {cls_name: 0 for cls_name in ALLOWED_CLASSES}

successful_images = 0
total_detections = 0


# PROCESS TARGET IMAGES

print()
print("=" * 70)
print(f"PROCESSING {TARGET_SIDE.upper()} PROFILE IMAGES (PER CLASS)")
print("=" * 70)

for index, image_path in enumerate(target_images, 1):
    print(f"[{index}/{len(target_images)}] {image_path.name}")

    image = cv2.imread(str(image_path))

    if image is None:
        print("  -> skipped: cannot read image")
        continue

    landmarks = detect_landmarks(image)

    if landmarks is None:
        print("  -> skipped: no face")
        del image
        continue

    orientation, nose_ratio = classify_profile(landmarks)

    print(f"  orientation = {orientation}")
    print(f"  nose ratio  = {nose_ratio:.4f}")

    if orientation != TARGET_SIDE:
        print(f"  -> skipped: not {TARGET_SIDE.upper()}")
        del image, landmarks
        continue

    try:
        aligned_image, matrix = align_image(
            image, landmarks, reference_landmarks,
            reference_width, reference_height,
        )
    except Exception as e:
        print("  -> skipped: alignment failed:", e)
        del image, landmarks
        continue

    # Detect acne on the original image grouped by class.
    detections = detect_acne(image, image_path)

    print(f"  acne detections = {len(detections)}")
    total_detections += len(detections)

    # Accumulate aligned face images.
    image_accumulator += aligned_image.astype(np.float32) * FACE_ALPHA

    # Project centers onto class-specific heatmaps.
    for class_name, box, confidence in detections:
        x1, y1, x2, y2 = box
        center = np.array([[(x1 + x2) * 0.5, (y1 + y2) * 0.5]], dtype=np.float32)
        aligned_center = transform_points(center, matrix)[0]
        x, y = aligned_center

        if 0 <= x < MAP_WIDTH and 0 <= y < MAP_HEIGHT:
            weight = confidence ** 2
            add_heat_point(class_heatmaps[class_name], x, y, weight, HEATMAP_SIGMA)
            class_counts[class_name] += 1

    successful_images += 1

    del image, landmarks, aligned_image, matrix, detections

    if index % 10 == 0:
        gc.collect()


# PRINT PROCESSING SUMMARY

print()
print("=" * 70)
print("PROCESSING SUMMARY (PER CLASS)")
print("=" * 70)

print(f"Images found:        {len(target_images)}")
print(f"Images aligned:      {successful_images}")
print(f"Total detections:    {total_detections}")
for cls_name, count in class_counts.items():
    print(f" - {cls_name} ({ALLOWED_CLASSES[cls_name]['pl']}): {count}")

if successful_images == 0:
    raise RuntimeError(f"Nie udało się wyrównać żadnego zdjęcia {TARGET_SIDE.upper()}.")


# CREATE AVERAGED FACE STACK

face_stack = image_accumulator / (FACE_ALPHA * successful_images)
face_stack = np.clip(face_stack, 0, 255).astype(np.uint8)

del image_accumulator
gc.collect()


# GENERATE & SAVE MAPS PER CLASS

print()
print("=" * 70)
print("GENERATING AND SAVING CLASS-SPECIFIC MAPS")
print("=" * 70)

for cls_name, raw_heatmap in class_heatmaps.items():
    count = class_counts[cls_name]
    pl_name = ALLOWED_CLASSES[cls_name]["pl"]

    # Normalizacja dla danej klasy
    heatmap_mean = raw_heatmap / successful_images
    nonzero_values = heatmap_mean[heatmap_mean > 0]

    if len(nonzero_values):
        heatmap_max = np.percentile(nonzero_values, 99)
    else:
        heatmap_max = 1.0

    heatmap_normalized = np.clip(heatmap_mean / max(heatmap_max, 1e-8), 0, 1)

    heatmap_uint8 = (heatmap_normalized * 255).astype(np.uint8)
    heatmap_color_bgr = cv2.applyColorMap(heatmap_uint8, cv2.COLORMAP_JET)

    heat_alpha = (heatmap_normalized * HEATMAP_ALPHA)[..., None]
    final_map = (
        face_stack.astype(np.float32) * (1.0 - heat_alpha)
        + heatmap_color_bgr.astype(np.float32) * heat_alpha
    )
    final_map = np.clip(final_map, 0, 255).astype(np.uint8)

    cls_map_path = MAP_OUTPUT_DIR / f"{SIDE_NAME}_map_{cls_name}.png"
    cls_npy_path = MAP_OUTPUT_DIR / f"{SIDE_NAME}_intensity_{cls_name}.npy"

    cv2.imwrite(str(cls_map_path), final_map)
    np.save(cls_npy_path, heatmap_normalized)

    print(f"Saved: {cls_map_path} (Wykryto: {count} szt.)")

    plt.figure(figsize=(7, 9))
    plt.imshow(cv2.cvtColor(final_map, cv2.COLOR_BGR2RGB))
    plt.title(
        f"{TARGET_SIDE.upper()} — Mapa: {pl_name} ({cls_name})\n"
        f"Liczba wykrytych zmian: {count} | n = {successful_images} twarzy"
    )
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    del heatmap_normalized, heatmap_color_bgr, final_map
    gc.collect()

print()
print("=" * 70)
print("ALL CLASS-SPECIFIC MAPS GENERATED SUCCESSFULLY")
print("=" * 70)